# EE2211 Tutorial 9 — Decision Trees

Tut 9 is four variations on the same theme. The notebook is **tools-first**: once the helpers below are defined, every question is a few lines of function calls.

| Tool | Use |
|---|---|
| `gini / entropy / misclass / node_metrics` | per-node impurity metrics (Q1) |
| `weighted_children / split_gain` | overall depth-1 metrics + information gain (Q1) |
| `mse / regression_split / find_best_split` | regression-tree MSE at any threshold (Q2, Q3) |
| `build_regression_tree / print_tree` | recursive tree up to any depth (Q3) |
| `fit_classification_tree` | train + score + plot a sklearn classifier (Q4) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.datasets import load_iris, fetch_california_housing
from sklearn.metrics import accuracy_score

## Tools — classification node metrics

Given class counts $[n_1, n_2, \dots, n_K]$ at a node (total $n = \sum n_k$), with $p_k = n_k/n$:

- **Gini impurity**: $G = 1 - \sum_k p_k^2$
- **Entropy** (Shannon, base 2): $H = -\sum_k p_k \log_2 p_k$ (with $0 \log 0 = 0$)
- **Misclassification rate**: $M = 1 - \max_k p_k$

For a parent split into children, the **overall metric** is the sample-weighted mean of the children's metrics:
$$\text{overall} = \sum_{c \in \text{children}} \frac{n_c}{n}\,\text{metric}(c).$$
**Information gain** / split quality = parent metric − overall metric (≥ 0 for Gini and entropy).

In [ ]:
def _to_arr(counts):
    return np.asarray(counts, dtype=float)

def gini(counts):
    c = _to_arr(counts); n = c.sum()
    if n == 0: return 0.0
    p = c / n
    return float(1.0 - np.sum(p ** 2))

def entropy(counts, base=2):
    c = _to_arr(counts); n = c.sum()
    if n == 0: return 0.0
    p = c[c > 0] / n                    # drop zeros so 0*log0 = 0
    return float(-np.sum(p * (np.log(p) / np.log(base))))

def misclass(counts):
    c = _to_arr(counts); n = c.sum()
    if n == 0: return 0.0
    return float(1.0 - c.max() / n)

def node_metrics(counts, label=""):
    c = _to_arr(counts)
    res = dict(n=int(c.sum()), counts=list(c.astype(int)),
               gini=gini(c), entropy=entropy(c), misclass=misclass(c))
    if label:
        print(f"{label:>4s}: n={res['n']:>3d}  counts={res['counts']}  "
              f"gini={res['gini']:.4f}  entropy={res['entropy']:.4f}  misclass={res['misclass']:.4f}")
    return res

def weighted_children(*child_counts):
    """Sample-weighted mean of gini/entropy/misclass across children."""
    totals = sum(sum(c) for c in child_counts)
    out = {'n': int(totals)}
    for key, fn in [('gini', gini), ('entropy', entropy), ('misclass', misclass)]:
        out[key] = sum(sum(c) * fn(c) for c in child_counts) / totals
    return out

def split_gain(parent_counts, *child_counts):
    p = node_metrics(parent_counts)
    w = weighted_children(*child_counts)
    return dict(
        parent=p, children=w,
        gini_gain    = p['gini']     - w['gini'],
        entropy_gain = p['entropy']  - w['entropy'],     # info gain
        misclass_gain= p['misclass'] - w['misclass'],
    )

# Smoke test — pure node should be zero impurity, uniform 50/50 → gini=0.5, H=1, misclass=0.5
print("pure        :", node_metrics([10, 0, 0]))
print("50/50       :", node_metrics([5, 5]))
print("uniform 3-cls:", node_metrics([4, 4, 4]))

## Question 1 — node metrics at depth 1 (Gini / entropy / misclass)

Read the class counts off the image for each node (order is arbitrary but must be consistent):
- **A** = parent node (all samples before the split)
- **B** = left child
- **C** = right child

Sanity check: the children's counts should sum per-class to the parent's counts.

> Counts below are my reading of the image; adjust the three lists if you count differently — the tools do the rest.

In [ ]:
# [circles, triangles, squares] — adjust to match your reading of the PDF image.
A = [7, 4, 4]     # parent
B = [4, 4, 0]     # left child
C = [3, 0, 4]     # right child

assert (np.array(B) + np.array(C)).tolist() == A, \
    f"children ({B}+{C}) don't sum to parent ({A}) — recheck your counts"

print("Per-node metrics:")
node_metrics(A, 'A')
node_metrics(B, 'B')
node_metrics(C, 'C')

print("\nDepth-1 overall (sample-weighted children) + gain vs parent:")
res = split_gain(A, B, C)
w = res['children']
print(f"  overall gini    = {w['gini']:.4f}   (gain = {res['gini_gain']:+.4f})")
print(f"  overall entropy = {w['entropy']:.4f}   (gain = {res['entropy_gain']:+.4f})")
print(f"  overall misclass= {w['misclass']:.4f}   (gain = {res['misclass_gain']:+.4f})")

## Tools — regression-tree MSE

For a leaf whose samples have target values $y$, the optimal constant prediction is $\bar y$ and the leaf MSE equals the **variance** of $y$:
$$\text{MSE}(y) = \frac{1}{n}\sum_{i}(y_i - \bar y)^2.$$
For a split on $x \le t$, the overall MSE is the sample-weighted mean of the two leaf MSEs:
$$\text{MSE}_\text{split} = \frac{n_L}{n}\,\text{MSE}(y_L) + \frac{n_R}{n}\,\text{MSE}(y_R).$$
`find_best_split` tries every midpoint between sorted unique $x$ values (what sklearn does) and picks the one with lowest overall MSE.

In [ ]:
def mse(y):
    y = np.asarray(y, dtype=float)
    if len(y) == 0: return 0.0
    return float(np.mean((y - y.mean()) ** 2))

def regression_split(x, y, threshold):
    """Split x<=t vs x>t; return per-side stats and the weighted (overall) MSE."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    L = y[x <= threshold]; R = y[x > threshold]
    n = len(y)
    w_mse = (len(L) * mse(L) + len(R) * mse(R)) / n if n else 0.0
    return dict(
        threshold=float(threshold),
        left =dict(n=len(L), mean=float(L.mean()) if len(L) else np.nan, mse=mse(L), y=L.tolist()),
        right=dict(n=len(R), mean=float(R.mean()) if len(R) else np.nan, mse=mse(R), y=R.tolist()),
        weighted_mse=float(w_mse),
        root_mse=mse(y),
    )

def find_best_split(x, y, candidates=None):
    """Try each threshold and return the split with lowest weighted MSE."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    if candidates is None:
        xs = np.sort(np.unique(x))
        candidates = (xs[:-1] + xs[1:]) / 2     # midpoints between consecutive unique values
    best = None
    for t in candidates:
        r = regression_split(x, y, t)
        if best is None or r['weighted_mse'] < best['weighted_mse']:
            best = r
    return best

def print_split(r, title=None):
    if title: print(title)
    L, R = r['left'], r['right']
    print(f"  threshold  = {r['threshold']}")
    print(f"  root MSE   = {r['root_mse']:.4f}")
    print(f"  left : n={L['n']:>2d}  mean={L['mean']:.4f}  MSE={L['mse']:.4f}")
    print(f"  right: n={R['n']:>2d}  mean={R['mean']:.4f}  MSE={R['mse']:.4f}")
    print(f"  overall (weighted) MSE = {r['weighted_mse']:.4f}")
    print(f"  MSE reduction          = {r['root_mse'] - r['weighted_mse']:+.4f}")

## Question 2 — regression-tree MSE with threshold $x=5.0$

Compute the overall MSE at depth 1 for the given $(x, y)$ data when the split is at $x=5.0$, and compare with the MSE at the root.

In [ ]:
q2 = [(1, 2), (0.8, 3), (2, 2.5), (2.5, 1), (3, 2.3), (4, 2.8), (4.2, 1.5),
      (6, 2.6), (6.3, 3.5), (7, 4), (8, 3.5), (8.2, 5), (9, 4.5)]
x = np.array([p[0] for p in q2])
y = np.array([p[1] for p in q2])

print_split(regression_split(x, y, threshold=5.0), title="Q2 split at x = 5.0")

# Bonus: where is the MSE-optimal split, and how much does it improve?
best = find_best_split(x, y)
print("\nBest split (sklearn-style midpoint search):")
print_split(best)

## Tools — recursive regression tree (for Q3)

Depth-2 manual fit reuses `find_best_split` twice: once at the root, once on each child. The recursion below scales to any depth.

In [ ]:
def build_regression_tree(x, y, max_depth, min_samples=1, depth=0):
    """Recursive regression tree on 1-D `x`. Returns a nested dict."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    node = dict(depth=depth, n=len(y),
                mean=float(y.mean()) if len(y) else np.nan,
                mse=mse(y), leaf=True)
    if depth >= max_depth or len(y) <= min_samples or np.unique(x).size < 2:
        return node
    b = find_best_split(x, y)
    # Skip pointless splits that can't reduce MSE
    if b is None or b['weighted_mse'] >= node['mse'] - 1e-12:
        return node
    mask = x <= b['threshold']
    node.update(leaf=False, threshold=b['threshold'],
                left =build_regression_tree(x[mask],  y[mask],  max_depth, min_samples, depth+1),
                right=build_regression_tree(x[~mask], y[~mask], max_depth, min_samples, depth+1))
    return node

def print_tree(node, indent=""):
    tag = "leaf" if node['leaf'] else f"x <= {node['threshold']:.4f}?"
    print(f"{indent}[d{node['depth']}] {tag}  n={node['n']}  mean={node['mean']:.4f}  mse={node['mse']:.4f}")
    if not node['leaf']:
        print_tree(node['left'],  indent + "  L ")
        print_tree(node['right'], indent + "  R ")

def predict_tree(node, x):
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    def _pred(n, xv):
        return n['mean'] if n['leaf'] else (_pred(n['left'], xv) if xv <= n['threshold'] else _pred(n['right'], xv))
    for i, xv in enumerate(x):
        out[i] = _pred(node, xv)
    return out

## Question 3 — regression tree on California Housing, depth 2

Fit to depth 2 using `MedInc` → `MedHouseVal`; compare our manual tree against sklearn's `DecisionTreeRegressor(criterion='squared_error', max_depth=2)`.

Both methods should agree on thresholds, leaf means, and per-leaf MSEs.

In [ ]:
housing = fetch_california_housing()
x_ca = housing.data[:, housing.feature_names.index('MedInc')]
y_ca = housing.target

print(f"N = {len(x_ca)}  root MSE = {mse(y_ca):.4f}  root mean = {y_ca.mean():.4f}\n")

print("=== Manual regression tree (depth 2) ===")
manual = build_regression_tree(x_ca, y_ca, max_depth=2)
print_tree(manual)

print("\n=== sklearn DecisionTreeRegressor(max_depth=2) ===")
reg = DecisionTreeRegressor(criterion='squared_error', max_depth=2, random_state=0)
reg.fit(x_ca.reshape(-1, 1), y_ca)

t = reg.tree_
print(f"  node_count = {t.node_count}")
for i in range(t.node_count):
    is_leaf = t.children_left[i] == -1
    if is_leaf:
        print(f"  node {i}: leaf  n={t.n_node_samples[i]:>5d}  mean={t.value[i, 0, 0]:.4f}  mse={t.impurity[i]:.4f}")
    else:
        print(f"  node {i}: MedInc <= {t.threshold[i]:.4f}?  n={t.n_node_samples[i]:>5d}  mean={t.value[i, 0, 0]:.4f}  mse={t.impurity[i]:.4f}")

# Predict with both and compare overall MSE
skl_mse = float(np.mean((reg.predict(x_ca.reshape(-1, 1)) - y_ca) ** 2))
man_mse = float(np.mean((predict_tree(manual, x_ca) - y_ca) ** 2))
print(f"\nWhole-tree MSE  manual = {man_mse:.6f}   sklearn = {skl_mse:.6f}   diff = {abs(man_mse - skl_mse):.2e}")

fig, ax = plt.subplots(figsize=(10, 4))
plot_tree(reg, feature_names=['MedInc'], filled=True, rounded=True, ax=ax)
plt.show()

## Tools — classification-tree wrapper (for Q4)

Thin helper around the `split → fit → score → plot` boilerplate. Works for any sklearn classification dataset.

In [ ]:
def fit_classification_tree(X, y, max_depth=4, criterion='entropy',
                            test_size=0.2, random_state=0,
                            feature_names=None, class_names=None,
                            plot=True, figsize=(14, 8)):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=random_state)
    clf = DecisionTreeClassifier(max_depth=max_depth, criterion=criterion, random_state=random_state)
    clf.fit(X_tr, y_tr)
    train_acc = accuracy_score(y_tr, clf.predict(X_tr))
    test_acc  = accuracy_score(y_te, clf.predict(X_te))

    print(f"criterion  = {criterion}")
    print(f"max_depth  = {max_depth}")
    print(f"Train acc  = {train_acc:.4f}")
    print(f"Test  acc  = {test_acc:.4f}")

    if plot:
        fig, ax = plt.subplots(figsize=figsize)
        plot_tree(clf, feature_names=feature_names, class_names=class_names,
                  filled=True, rounded=True, ax=ax)
        plt.show()

    return dict(clf=clf, train_acc=train_acc, test_acc=test_acc,
                X_train=X_tr, X_test=X_te, y_train=y_tr, y_test=y_te)

## Question 4 — Iris decision tree, entropy, depth 4

80 / 20 split with `random_state=0`, entropy criterion, `max_depth=4`; report train/test accuracies and plot the tree.

In [ ]:
iris = load_iris()
out = fit_classification_tree(
    iris.data, iris.target,
    max_depth=4, criterion='entropy',
    test_size=0.2, random_state=0,
    feature_names=iris.feature_names,
    class_names=list(iris.target_names),
)

## Takeaways

- **Impurity formulas collapse to class probabilities** — $p_k = n_k/n$ is all you need; every metric (Gini / entropy / misclass) is a one-liner over the $p_k$ vector.
- **Depth-1 "overall metric"** = sample-weighted mean of the children's metrics. Gain = parent metric − overall.
- **Regression-tree MSE** is just the variance of $y$ at a node; sklearn chooses splits at midpoints of sorted unique $x$ to minimize weighted MSE across the two children.
- **Manual tree ↔ sklearn** — thresholds, leaf means, and per-leaf MSEs match exactly when both use the squared-error criterion (Q3).
- **Entropy vs Gini in practice** — usually pick very similar splits; the exam often asks for one or the other to test the formula, not the choice.